# Qwen2.5 3B Instruct — Green Computing Token Experiment

Third model run for the prompt-optimization study.

| Setting | Value |
|---|---|
| Model | `qwen2.5:3b-instruct` |
| Hardware | Google Colab **T4 GPU** |
| Temperature | 0 |
| Seed | 42 |
| Runs | 200 (100 baseline + 100 optimized) |
| Output | `data/raw/runs_qwen25_3b_colab.jsonl` |

**Run the cells in order.** Every cell is safe to re-run; the experiment
runner skips work that is already complete.

Do not edit the prompts or context files. Do not delete `data/raw/runs.jsonl`
(the finished Gemma run) or any Llama result file.

## 1. Confirm the T4 GPU

Stops immediately if the runtime is not a GPU runtime.

In [1]:
import subprocess, sys

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(out.stdout or out.stderr)

if out.returncode != 0:
    sys.exit("No GPU detected. Set Runtime -> Change runtime type -> T4 GPU, then rerun.")
if "T4" not in out.stdout:
    print("WARNING: a GPU is present but it is not a T4. "
          "Record the actual GPU name in your handover notes.")

Fri Sep 18 14:31:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

Drive keeps the results if the Colab session resets.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Clone the project into Drive

In [3]:
import os

REPO_URL = "https://github.com/siifat/green-llm-token-research.git"
PROJECT = "/content/drive/MyDrive/green-llm-token-research"

if not os.path.exists(PROJECT):
    !git clone "$REPO_URL" "$PROJECT"
else:
    print("Project folder already exists. Not cloning again.")

%cd "$PROJECT"

Project folder already exists. Not cloning again.
/content/drive/MyDrive/green-llm-token-research


## 4. Verify the dataset

Hard-stops if the dataset is not exactly the shared 100-pair set, or if any
context file referenced by the CSV is missing.

In [4]:
from pathlib import Path
import csv, sys

bundle = Path("green_llm_ready_bundle")
csv_path = bundle / "green_llm_ready_100_prompts.csv"
assert csv_path.exists(), f"Dataset not found: {csv_path}"

with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

print("Dataset exists:", csv_path.exists())
print("Number of prompt pairs:", len(rows))

p3 = [r for r in rows if r["prompt_id"] == "P0003"][0]
print("P0003 source pair:", p3["source_pair_id"])
print("P0003 task:", p3["task"])

missing = [
    rel
    for r in rows
    for col in ("baseline_context_file", "optimized_context_file")
    for rel in [(r.get(col) or "").strip()]
    if rel and not (bundle / rel).exists()
]

assert len(rows) == 100, f"Expected 100 prompt pairs, found {len(rows)}. STOP."
assert p3["source_pair_id"] == "1.26", f"P0003 is {p3['source_pair_id']}, expected 1.26. STOP."
assert not missing, f"Missing context files: {missing[:5]}. STOP."

print(f"\nAll dataset checks passed ({len(missing)} missing context files).")

Dataset exists: True
Number of prompt pairs: 100
P0003 source pair: 1.26
P0003 task: Write the discussion of a net present value appraisal for a finance assignment.

All dataset checks passed (0 missing context files).


## 5. Install Ollama

The Ollama installer ships a **zstd**-compressed archive, and the Colab image
does not include `zstd`. Installing it first avoids:

```
ERROR: This version requires zstd for extraction.
```

In [5]:
# Colab runs as root, so no sudo is needed. Retry with a package-list refresh
# in case the image's apt metadata is stale.
!apt-get install -y -qq zstd || (apt-get update -qq && apt-get install -y -qq zstd)
!zstd --version

Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
*** Zstandard CLI (64-bit) v1.5.5, by Yann Collet ***


In [6]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
import shutil, sys
if shutil.which("ollama") is None:
    sys.exit("Ollama did not install. Check the zstd cell above completed successfully.")
print("Ollama binary found at:", shutil.which("ollama"))

Ollama binary found at: /usr/local/bin/ollama


## 6. Start the Ollama server

Models are stored in Drive so they survive a session reset. The cell polls
until the server actually answers rather than sleeping a fixed time.

In [8]:
import os, subprocess, time, urllib.request, sys

os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/ollama_models"
os.makedirs(os.environ["OLLAMA_MODELS"], exist_ok=True)

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

for attempt in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2).read()
        print(f"Ollama server responding after {attempt + 1}s.")
        break
    except Exception:
        time.sleep(1)
else:
    print(open("/tmp/ollama.log").read())
    sys.exit("Ollama server did not start. See the log above.")

!ollama list

Ollama server responding after 3s.
NAME                   ID              SIZE      MODIFIED       
qwen2.5:3b-instruct    357c53fb659c    1.9 GB    27 minutes ago    


## 7. Pull Qwen2.5 3B Instruct

About 2 GB. Slower than usual because it is written to Drive.

In [9]:
!ollama pull qwen2.5:3b-instruct
!ollama run qwen2.5:3b-instruct "Reply with exactly: READY"
!ollama ps


READY

NAME                   ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen2.5:3b-instruct    357c53fb659c    2.2 GB    100% GPU     4096       4 minutes from now    


## 8. Create the Qwen runner

Copies the existing runner and swaps three values. **Each replacement must
match exactly once** — the assertion below is important: if a replacement
silently matched nothing, the copy would still say `gemma3:4b` and still write
to `data/raw/runs.jsonl`, appending junk to the finished Gemma dataset.

In [10]:
from pathlib import Path

source = Path("scripts/run_experiment.py")
target = Path("scripts/run_experiment_qwen25_colab.py")
text = source.read_text(encoding="utf-8")

substitutions = [
    ('MODEL = "gemma3:4b"',
     'MODEL = "qwen2.5:3b-instruct"'),
    ('output_path = project_root / "data" / "raw" / "runs.jsonl"',
     'output_path = project_root / "data" / "raw" / "runs_qwen25_3b_colab.jsonl"'),
    ('error_path = project_root / "logs" / "experiment_errors.jsonl"',
     'error_path = project_root / "logs" / "experiment_errors_qwen25_3b_colab.jsonl"'),
]

for old, new in substitutions:
    n = text.count(old)
    assert n == 1, f"Expected exactly 1 occurrence of {old!r}, found {n}. STOP - do not run."
    text = text.replace(old, new)

target.write_text(text, encoding="utf-8")
print("Created:", target)

written = target.read_text(encoding="utf-8")
assert 'MODEL = "qwen2.5:3b-instruct"' in written
assert 'runs_qwen25_3b_colab.jsonl' in written
assert 'data" / "raw" / "runs.jsonl"' not in written, "Runner still targets the Gemma output file. STOP."
print("Verified: correct model and output path; Gemma output untouched.")

Created: scripts/run_experiment_qwen25_colab.py
Verified: correct model and output path; Gemma output untouched.


In [11]:
!grep -n 'MODEL =' scripts/run_experiment_qwen25_colab.py
!grep -n 'runs_qwen25_3b_colab\|experiment_errors_qwen25' scripts/run_experiment_qwen25_colab.py

11:MODEL = "qwen2.5:3b-instruct"
151:    output_path = project_root / "data" / "raw" / "runs_qwen25_3b_colab.jsonl"
152:    error_path = project_root / "logs" / "experiment_errors_qwen25_3b_colab.jsonl"


## 9. Pilot run (P0001 only)

Runs P0001 baseline and optimized to confirm the pipeline works.

In [12]:
!python scripts/run_experiment_qwen25_colab.py --limit 1

Model: qwen2.5:3b-instruct
Tasks in scope: 1
Expected runs: 2
Already completed: 2
Remaining: 0
Output: /content/drive/MyDrive/green-llm-token-research/data/raw/runs_qwen25_3b_colab.jsonl

Nothing to do. All runs in scope are already complete.


## 10. Delete only the pilot output

Removes the two Qwen pilot files so the full run starts clean.
Leaves `data/raw/runs.jsonl` and any Llama file untouched.

In [13]:
from pathlib import Path

for p in [
    Path("data/raw/runs_qwen25_3b_colab.jsonl"),
    Path("logs/experiment_errors_qwen25_3b_colab.jsonl"),
]:
    if p.exists():
        p.unlink()
        print("Deleted pilot file:", p)

assert Path("data/raw/runs.jsonl").exists(), "Gemma results missing - do not continue."
print("Gemma results intact.")

Deleted pilot file: data/raw/runs_qwen25_3b_colab.jsonl
Gemma results intact.


## 11. Full experiment — 200 runs

Expect roughly 30-60 minutes on a T4. (Gemma 3 4B took 2.65 hours for the same
200 runs on a desktop CPU, so a GPU should be considerably faster.)

The runner saves after every run. If Colab disconnects, rerun this same cell
after redoing cells 2, 3, 5, 6 and 7 — completed runs are skipped.

In [14]:
!python scripts/run_experiment_qwen25_colab.py

Model: qwen2.5:3b-instruct
Tasks in scope: 100
Expected runs: 200
Already completed: 0
Remaining: 200
Output: /content/drive/MyDrive/green-llm-token-research/data/raw/runs_qwen25_3b_colab.jsonl

Warming up the model...
Warm-up complete.

[1/100] P0001 - baseline ... done | in=114 out=492 total=606 time=6.57s
[1/100] P0001 - optimized ... done | in=82 out=291 total=373 time=3.95s
[2/100] P0002 - optimized ... done | in=94 out=440 total=534 time=6.10s
[2/100] P0002 - baseline ... done | in=109 out=1061 total=1170 time=15.83s
[3/100] P0003 - baseline ... done | in=109 out=382 total=491 time=6.00s
[3/100] P0003 - optimized ... done | in=88 out=398 total=486 time=6.44s
[4/100] P0004 - optimized ... done | in=74 out=207 total=281 time=3.45s
[4/100] P0004 - baseline ... done | in=95 out=160 total=255 time=2.75s
[5/100] P0005 - baseline ... done | in=94 out=210 total=304 time=3.66s
[5/100] P0005 - optimized ... done | in=82 out=178 total=260 time=3.17s
[6/100] P0006 - optimized ... done | in=8

## 12. Validate and summarise

Covers handoff steps 14-19. Exits non-zero if any check fails.

In [15]:
!python scripts/validate_qwen25_colab.py

python3: can't open file '/content/drive/MyDrive/green-llm-token-research/scripts/validate_qwen25_colab.py': [Errno 2] No such file or directory


## 13. Copy out what the team asked for

Send back:

1. `data/raw/runs_qwen25_3b_colab.jsonl`
2. The record count (200)
3. The variant split (100 baseline + 100 optimized)
4. The Qwen summary block
5. `logs/experiment_errors_qwen25_3b_colab.jsonl` if it exists
6. A note on anything that failed repeatedly or needed manual intervention

In [16]:
from pathlib import Path

result = Path("data/raw/runs_qwen25_3b_colab.jsonl")
errors = Path("logs/experiment_errors_qwen25_3b_colab.jsonl")

print("Result file:", result.resolve(), f"({result.stat().st_size:,} bytes)" if result.exists() else "MISSING")
print("Error log:", f"{errors.resolve()} ({errors.stat().st_size:,} bytes)" if errors.exists()
      else "none created (no failures)")
print("\nBoth live in your Google Drive under green-llm-token-research/.")

Result file: /content/drive/MyDrive/green-llm-token-research/data/raw/runs_qwen25_3b_colab.jsonl (1,383,175 bytes)
Error log: none created (no failures)

Both live in your Google Drive under green-llm-token-research/.
